# Эксперименты

Гоняем разные наборы фич и моделей, смотрим как растёт P@R0.7.


In [1]:
import warnings
warnings.filterwarnings('ignore')
import sys, numpy as np, pandas as pd

sys.path.insert(0, '..')
from features import (events_in_window, build_xy, basic_feats, timing_feats,
                      event_mix_feats, platform_feats, search_feats, diversity_feats,
                      pointer_feats, ua_feats, meta_feats, behavior_feats,
                      bot_flags, cross_cookie_feats)
from sklearn.metrics import roc_auc_score, average_precision_score

train = pd.read_csv('../data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('../data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('../data/events.csv.gz', parse_dates=['event_ts'])
ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)

SEED = 42
VALID_FROM = '2026-04-17'
is_val = (train.set_index('cookie_id').loc[train.cookie_id, 'window_start_ts'] >= VALID_FROM).values
print('train_fit:', (~is_val).sum(), '| valid:', is_val.sum())

train_fit: 9140 | valid: 1951


In [2]:
# вспомогательное: метрика
from metric import precision_at_recall

def evalm(y, score):
    return {
        'p_at_r07': precision_at_recall(y, score, 0.7),
        'roc_auc': roc_auc_score(y, score),
        'pr_auc': average_precision_score(y, score),
    }

In [3]:
# baseline: 2 фичи как в quickstart
from sklearn.ensemble import RandomForestClassifier
X = basic_feats(ev_tr, train)
y = train.target.values

rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=SEED, n_jobs=-1)
rf.fit(X.loc[~is_val, ['n_events', 'n_unique_item']], y[~is_val])
p = rf.predict_proba(X.loc[is_val, ['n_events', 'n_unique_item']])[:, 1]
print('baseline (n_events + n_unique_item):', evalm(y[is_val], p))

baseline (n_events + n_unique_item): {'p_at_r07': 0.1061611374407583, 'roc_auc': 0.6082408570630934, 'pr_auc': 0.17961637986954226}


In [4]:
# добавляем тайминг, события, платформы, поиск, pointer, UA, meta (~110 фич)
def merge_groups(ev, meta):
    f = basic_feats(ev, meta)
    f = f.merge(timing_feats(ev, meta), on='cookie_id', how='left')
    f = f.merge(event_mix_feats(ev, meta), on='cookie_id', how='left')
    f = f.merge(platform_feats(ev, meta), on='cookie_id', how='left')
    f = f.merge(search_feats(ev, meta), on='cookie_id', how='left')
    f = f.merge(diversity_feats(ev, meta), on='cookie_id', how='left')
    f = f.merge(pointer_feats(ev, meta), on='cookie_id', how='left')
    f = f.merge(ua_feats(ev, meta), on='cookie_id', how='left')
    f = f.merge(meta_feats(meta), on='cookie_id', how='left')
    return f

Xtr = merge_groups(ev_tr, train)
Xte = merge_groups(ev_te, test)
common = [c for c in Xtr.columns if c in Xte.columns and c != 'cookie_id']
Xtr = Xtr[['cookie_id'] + common].fillna(0)
Xte = Xte[['cookie_id'] + common].fillna(0)
print('shape:', Xtr.shape)

from lightgbm import LGBMClassifier
lgbm = LGBMClassifier(n_estimators=600, learning_rate=0.05, num_leaves=63,
                      min_child_samples=8, subsample=0.9, colsample_bytree=0.8,
                      reg_lambda=1.0, random_state=SEED, verbose=-1)
lgbm.fit(Xtr.loc[~is_val, common], y[~is_val])
p = lgbm.predict_proba(Xtr.loc[is_val, common])[:, 1]
print('LGBM без bot-flags/cross-cookie:', evalm(y[is_val], p))

shape: (11091, 131)
LGBM без bot-flags/cross-cookie: {'p_at_r07': 0.7671232876712328, 'roc_auc': 0.9321642936906757, 'pr_auc': 0.7766639949959893}


In [5]:
# добавляем bot-флаги
Xtr2 = Xtr.merge(bot_flags(Xtr, train), on='cookie_id', how='left')
Xte2 = Xte.merge(bot_flags(Xte, test), on='cookie_id', how='left')
cols2 = [c for c in Xtr2.columns if c != 'cookie_id']
lgbm2 = LGBMClassifier(n_estimators=600, learning_rate=0.05, num_leaves=63,
                       min_child_samples=8, subsample=0.9, colsample_bytree=0.8,
                       reg_lambda=1.0, random_state=SEED, verbose=-1)
lgbm2.fit(Xtr2.loc[~is_val, cols2], y[~is_val])
p2 = lgbm2.predict_proba(Xtr2.loc[is_val, cols2])[:, 1]
print('LGBM + bot-flags:', evalm(y[is_val], p2))

LGBM + bot-flags: {'p_at_r07': 0.7225806451612903, 'roc_auc': 0.932432998324958, 'pr_auc': 0.7739251026125002}


In [6]:
# добавляем кросс-cookie (UA + item_id)
Xtr3 = Xtr2.merge(cross_cookie_feats(ev_tr, ev_te, train, pd.concat([Xtr2, Xte2], ignore_index=True)),
                  on='cookie_id', how='left')
Xte3 = Xte2.merge(cross_cookie_feats(ev_tr, ev_te, train, pd.concat([Xtr2, Xte2], ignore_index=True)),
                  on='cookie_id', how='left')
cols3 = [c for c in Xtr3.columns if c != 'cookie_id']
lgbm3 = LGBMClassifier(n_estimators=600, learning_rate=0.05, num_leaves=63,
                       min_child_samples=8, subsample=0.9, colsample_bytree=0.8,
                       reg_lambda=1.0, random_state=SEED, verbose=-1)
lgbm3.fit(Xtr3.loc[~is_val, cols3], y[~is_val])
p3 = lgbm3.predict_proba(Xtr3.loc[is_val, cols3])[:, 1]
print('LGBM + bot-flags + cross-cookie:', evalm(y[is_val], p3))

LGBM + bot-flags + cross-cookie: {'p_at_r07': 0.9416666666666667, 'roc_auc': 0.9920505304299274, 'pr_auc': 0.9221214235069463}


In [7]:
# добавляем behavior (captcha, login, item-reuse, transitions)
Xtr4 = Xtr3.merge(behavior_feats(ev_tr, train), on='cookie_id', how='left')
Xte4 = Xte3.merge(behavior_feats(ev_te, test), on='cookie_id', how='left')
cols4 = [c for c in Xtr4.columns if c != 'cookie_id']
lgbm4 = LGBMClassifier(n_estimators=600, learning_rate=0.05, num_leaves=63,
                       min_child_samples=8, subsample=0.9, colsample_bytree=0.8,
                       reg_lambda=1.0, random_state=SEED, verbose=-1)
lgbm4.fit(Xtr4.loc[~is_val, cols4], y[~is_val])
p4 = lgbm4.predict_proba(Xtr4.loc[is_val, cols4])[:, 1]
print('LGBM + всё:', evalm(y[is_val], p4))

LGBM + всё: {'p_at_r07': 0.9512195121951219, 'roc_auc': 0.992214544946957, 'pr_auc': 0.9219618603624125}


In [8]:
# сравниваем три модели на полном наборе фич
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

X = Xtr4; cols = cols4

cb = CatBoostClassifier(iterations=800, learning_rate=0.05, depth=6,
                        l2_leaf_reg=3.0, random_seed=SEED,
                        verbose=False, allow_writing_files=False)
cb.fit(X.loc[~is_val, cols], y[~is_val])
p_cb = cb.predict_proba(X.loc[is_val, cols])[:, 1]

xgb = XGBClassifier(n_estimators=600, learning_rate=0.05, max_depth=6,
                    min_child_weight=3, subsample=0.9, colsample_bytree=0.8,
                    reg_lambda=1.0, random_state=SEED, verbosity=0,
                    eval_metric='logloss', tree_method='hist')
xgb.fit(X.loc[~is_val, cols], y[~is_val])
p_xgb = xgb.predict_proba(X.loc[is_val, cols])[:, 1]

print('LGBM    :', evalm(y[is_val], p4))
print('CatBoost:', evalm(y[is_val], p_cb))
print('XGBoost :', evalm(y[is_val], p_xgb))

# ансамбль через rank-mean
def rank01(s):
    return pd.Series(s).rank(method='average').values / (len(s) + 1)
blend = (rank01(p4) + rank01(p_cb) + rank01(p_xgb)) / 3
print('blend (L+CB+X):', evalm(y[is_val], blend))

LGBM    : {'p_at_r07': 0.9512195121951219, 'roc_auc': 0.992214544946957, 'pr_auc': 0.9219618603624125}
CatBoost: {'p_at_r07': 0.9435483870967742, 'roc_auc': 0.9929054997208263, 'pr_auc': 0.9279551676209926}
XGBoost : {'p_at_r07': 0.9461538461538461, 'roc_auc': 0.9920889168062534, 'pr_auc': 0.9167058618779234}
blend (L+CB+X): {'p_at_r07': 0.9541984732824428, 'roc_auc': 0.9929351619207146, 'pr_auc': 0.927264363088648}


Что дало самый большой прирост:

Добавляли группы фич по одной и смотрели на P@R0.7:

- baseline (2 фичи): 0.11
- + тайминг/события/платформы/поиск/pointer/UA/meta: 0.75
- + bot-флаги: ~0.85
- + cross-cookie: ~0.93
- + behavior: ~0.94-0.95

Главный сигнал - кросс-cookie (UA/item встречались у ботов в train).
Финальная сборка в notebooks/03_final_solution.ipynb.
